# S2.1 · 合成数据生成与正确性验收

本步用结构因果模型（SCM）生成纵向联邦场景的双方数据。**关键在于地面真值已知**：互补性 λ、冗余度、重叠率都是我们设定的，因此可以检验模型学到的增益是否与理论增益一致。

> ⚠️ 合成数据只能验证**机制**，不能替代真实业务数据做效果承诺。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m2_synthetic/configs/scenarios.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m2_synthetic/configs/scenarios.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 7f29b9e
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
from modules.m2_synthetic.components.scm_generator import (
    load_scenarios, generate, usable_mask, theoretical_gain_signal)
scenarios = load_scenarios(config)
pd.DataFrame([vars(s) for s in scenarios]).set_index('name')

,n_party_a,overlap_rate,complementarity,redundancy,marginal_drift,size_asymmetry,base_rate,consent_rate,consent_selectivity,match_error_rate,time_drift,label_noise,hte_strength,dim_shared,dim_private_a,dim_private_b
name,,,,,,,,,,,,,,,,
S1_基准,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,4,6,6
S2_零互补,40000,0.3,0.0,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,4,6,6
S3_高互补,40000,0.3,2.0,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,4,6,6
S4_高冗余,40000,0.3,0.8,1.2,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,4,6,6
S5_低重叠高漂移,40000,0.1,0.8,0.3,0.8,1.0,0.03,0.6,0.0,0.00,1.0,0.0,0.5,4,6,6
S6_匹配噪声,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.6,0.0,0.05,0.0,0.0,0.5,4,6,6
S7_同意选择偏差,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.4,1.0,0.00,0.0,0.0,0.5,4,6,6
S8_稀疏正样本,40000,0.3,1.2,0.3,0.0,1.0,0.01,0.6,0.0,0.00,0.0,0.0,0.5,4,6,6


## 各场景的可用样本量与合规折损

`usable = 交集 ∩ 同意`。这一步的折损是**合规成本的直接体现**。

In [3]:
rows = []
for s in scenarios:
    d = generate(s, seed)
    m = usable_mask(d)
    rows.append({'场景': s.name, '总样本': len(d['y']),
                 '交集内': int(d['in_overlap'].sum()),
                 '交集且同意': int(m.sum()),
                 '正样本率': float(d['y_control'][m].mean()),
                 '合规留存率': float(m.mean())})
pd.DataFrame(rows).set_index('场景').round(ROUND_DP)

,总样本,交集内,交集且同意,正样本率,合规留存率
场景,,,,,
S1_基准,40000,11882,7236,0.0258,0.1809
S2_零互补,40000,11882,7236,0.0286,0.1809
S3_高互补,40000,11882,7236,0.0250,0.1809
S4_高冗余,40000,11882,7236,0.0258,0.1809
S5_低重叠高漂移,40000,3993,2469,0.0255,0.0617
S6_匹配噪声,40000,11882,7236,0.0258,0.1809
S7_同意选择偏差,40000,11882,4799,0.0502,0.1200
S8_稀疏正样本,40000,11882,7236,0.0084,0.1809


## 正确性验收：理论增益 vs 实测增益

`theoretical_gain_signal` 用**真实信号**构造两个 oracle 打分（含 B / 不含 B），其 AUC 差就是该场景下 B 侧数据的理论价值上界。

In [4]:
from sklearn.metrics import roc_auc_score
rows = []
for s in scenarios:
    for sd in config['seeds']:
        d = generate(s, sd); m = usable_mask(d); g = theoretical_gain_signal(d)
        y = d['y_control'][m]
        rows.append({'场景': s.name, '种子': sd,
                     '理论增益': roc_auc_score(y, g['with_b'][m])
                                 - roc_auc_score(y, g['without_b'][m])})
th = pd.DataFrame(rows).groupby('场景')['理论增益'].agg(['mean', 'std']).round(ROUND_DP)
th

,mean,std
场景,,
S1_基准,0.0360,0.0050
S2_零互补,0.0000,0.0000
S3_高互补,0.1534,0.0119
S4_高冗余,0.0360,0.0050
S5_低重叠高漂移,0.0355,0.0095
S6_匹配噪声,0.0360,0.0050
S7_同意选择偏差,0.0468,0.0031
S8_稀疏正样本,0.0778,0.0238


**验收判据**：S2_零互补 的理论增益必须为 0——这是生成器实现正确的硬检验。

In [5]:
ZERO_TOL = 1e-9
v = float(th.loc['S2_零互补', 'mean'])
print('S2_零互补 理论增益 =', v)
print('验收:', '通过' if abs(v) < ZERO_TOL else '不通过')

S2_零互补 理论增益 = 0.0
验收: 通过
